In [1]:
import platform
arch = platform.machine()
print(arch)

arm64


In [1]:
%pip install -q mlx-lm huggingface_hub datasets transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from getpass import getpass
from huggingface_hub import login

HF_TOKEN = getpass("Paste your Hugging Face token (starts with hf_): ").strip()
if not HF_TOKEN:
    raise ValueError("No token provided. Generate one at https://huggingface.co/settings/tokens")
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("That does not look like a valid Hugging Face token.")

login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged in to Hugging Face Hub")

In [ ]:
from pathlib import Path
import re

ROOT = Path("/Users/shreeyansvichare/Code/ML/SurvivorLM")
FINE_TUNE_DIR = ROOT / "fine-tuning"
DATA_DIR = FINE_TUNE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.jsonl"
VALID_FILE = DATA_DIR / "valid.jsonl"
TRAIN_SPLIT = 0.9
SEED = 42
MODEL_ID = "google/gemma-4-e2b-it"

SOURCE_DIRS = [
    ROOT / "data" / "dataset",
    ROOT / "data" / "extracted",
    ROOT / "data" / "extracted2",
    ROOT / "more" / "instructions",
    ROOT / "more" / "warmup",
    FINE_TUNE_DIR,  # includes instructions_upsampled.txt
]

print("Model:", MODEL_ID)
print("Output data dir:", DATA_DIR)

In [ ]:
MAX_Q_CHARS = 350
MAX_A_CHARS = 1400

def clean_text(s: str) -> str:
    s = s.replace("\u0000", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_long_text(s: str, max_chars: int = MAX_A_CHARS):
    if len(s) <= max_chars:
        return [s]
    chunks = []
    start = 0
    while start < len(s):
        end = min(start + max_chars, len(s))
        # prefer to split near sentence boundary
        if end < len(s):
            cut = s.rfind(". ", start, end)
            if cut > start + 200:
                end = cut + 1
        chunks.append(s[start:end].strip())
        start = end
    return [c for c in chunks if len(c) >= 80]

def chunk_paragraphs(text: str, max_chars: int = 1200):
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, current = [], ""
    for p in paragraphs:
        candidate = f"{current}\n\n{p}" if current else p
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            if len(p) <= max_chars:
                current = p
            else:
                for seg in split_long_text(p, max_chars=max_chars):
                    chunks.append(seg)
                current = ""
    if current:
        chunks.append(current)
    return chunks

def extract_pairs_from_file(path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    text = text.replace("\r\n", "\n")
    pairs = []

    # 1) Q/A formatted data
    qa_pattern = re.compile(r"Q:\s*(.+?)\nA:\s*(.+?)(?=\nQ:|\Z)", re.DOTALL | re.IGNORECASE)
    for m in qa_pattern.finditer(text):
        q = clean_text(m.group(1))[:MAX_Q_CHARS]
        a = clean_text(m.group(2))
        if len(q) < 8 or len(a) < 20:
            continue
        for a_chunk in split_long_text(a, max_chars=MAX_A_CHARS):
            pairs.append({"question": q, "answer": a_chunk})

    # 2) Prose/manual text -> instruction style pairs
    if len(pairs) == 0:
        topic = path.stem.replace("_", " ").replace("-", " ").strip()
        for chunk in chunk_paragraphs(text, max_chars=MAX_A_CHARS):
            chunk = clean_text(chunk)
            if len(chunk) < 120:
                continue
            question = f"Teach me practical survival guidance about {topic}."
            pairs.append({"question": question, "answer": chunk})

    return pairs

all_txt_files = []
for d in SOURCE_DIRS:
    if d.exists():
        all_txt_files.extend(sorted(d.rglob("*.txt")))

exclude_names = {"train.jsonl", "valid.jsonl", "training.log"}
all_txt_files = [p for p in all_txt_files if p.name not in exclude_names]

pairs = []
for p in all_txt_files:
    pairs.extend(extract_pairs_from_file(p))

# Deduplicate
seen, deduped = set(), []
for item in pairs:
    key = (item["question"].lower(), item["answer"].lower())
    if key not in seen:
        seen.add(key)
        deduped.append(item)
pairs = deduped

print(f"✅ Found {len(all_txt_files)} source files")
print(f"✅ Built {len(pairs)} training pairs")
print("\nSample:")
print("Q:", pairs[0]["question"][:140])
print("A:", pairs[0]["answer"][:240], "...")

In [ ]:
import json
import random
from transformers import AutoTokenizer

SYSTEM_PROMPT = (
    "You are SurvivalGuide, an expert survival assistant. "
    "Give practical, concise, safety-first advice for wilderness survival, first aid, "
    "disaster response, hydration, shelter, signaling, and preparedness."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def format_for_mlx(pair):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pair["question"]},
        {"role": "assistant", "content": pair["answer"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

random.seed(SEED)
random.shuffle(pairs)
split_idx = int(len(pairs) * TRAIN_SPLIT)
train_pairs = pairs[:split_idx]
valid_pairs = pairs[split_idx:]

def write_jsonl(records, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(format_for_mlx(r), ensure_ascii=False) + "\n")

write_jsonl(train_pairs, TRAIN_FILE)
write_jsonl(valid_pairs, VALID_FILE)

print(f"✅ Train: {len(train_pairs)} examples -> {TRAIN_FILE}")
print(f"✅ Valid: {len(valid_pairs)} examples -> {VALID_FILE}")

with open(TRAIN_FILE, encoding="utf-8") as f:
    sample = json.loads(f.readline())
print("\nFormatted sample (truncated):")
print(sample["text"][:400] + "...")

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "google/gemma-4-e2b-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

lengths = []
with open(TRAIN_FILE) as f:
    for line in f:
        record = json.loads(line)
        tokens = tokenizer(record["text"], return_length=True)["length"][0]
        lengths.append(tokens)

import statistics
print(f"Token length stats (train set):")
print(f"  Min:    {min(lengths)}")
print(f"  Max:    {max(lengths)}")
print(f"  Mean:   {statistics.mean(lengths):.0f}")
print(f"  Median: {statistics.median(lengths):.0f}")
print(f"  P95:    {sorted(lengths)[int(0.95*len(lengths))]}")
print("\n💡 Set --max-tokens to P95 value for efficient training.")

In [ ]:
# ── TRAINING CONFIG ───────────────────────────────────────────────────────────
ADAPTER_DIR   = FINE_TUNE_DIR / "adapters" / "survival-gemma4-2b"
LOG_FILE      = FINE_TUNE_DIR / "training.log"
ITERS         = 1200
BATCH_SIZE    = 2
LORA_LAYERS   = 12
LEARNING_RATE = "8e-6"
MAX_TOKENS    = 512  # based on observed P95 token length
SAVE_EVERY    = 100
# ─────────────────────────────────────────────────────────────────────────────

import os
import shlex
import sys

os.makedirs(ADAPTER_DIR, exist_ok=True)

cmd = (
    f"{shlex.quote(sys.executable)} -m mlx_lm lora "
    f"--model {shlex.quote(MODEL_ID)} "
    f"--train "
    f"--data {shlex.quote(str(DATA_DIR))} "
    f"--fine-tune-type lora "
    f"--mask-prompt "
    f"--adapter-path {shlex.quote(str(ADAPTER_DIR))} "
    f"--iters {ITERS} "
    f"--batch-size {BATCH_SIZE} "
    f"--num-layers {LORA_LAYERS} "
    f"--learning-rate {LEARNING_RATE} "
    f"--max-seq-length {MAX_TOKENS} "
    f"--steps-per-eval 50 "
    f"--steps-per-report 25 "
    f"--save-every {SAVE_EVERY} "
    f"--val-batches 25 "
    f"--grad-checkpoint"
)

print("🚀 Starting LoRA fine-tuning...")
print(f"Command:\n{cmd}\n")
print(f"Logs -> {LOG_FILE}\n")

!{cmd} 2>&1 | tee {LOG_FILE}

In [ ]:
# MLX outputs loss logs to stdout; this cell plots them if you've captured them.
# If running interactively, paste the training output into `log_text` below.

import re
import matplotlib.pyplot as plt

# Example: parse from a saved log file if you redirected output
LOG_FILE = "training.log"  # optional — run Cell 6 with `!{cmd} | tee training.log`

try:
    with open(LOG_FILE) as f:
        log_text = f.read()

    steps, train_losses, val_losses = [], [], []

    for m in re.finditer(r"Iter (\d+): Train loss (\d+\.\d+)", log_text):
        steps.append(int(m.group(1)))
        train_losses.append(float(m.group(2)))

    for m in re.finditer(r"Val loss (\d+\.\d+)", log_text):
        val_losses.append(float(m.group(1)))

    plt.figure(figsize=(10, 4))
    plt.plot(steps, train_losses, label="Train Loss", color="steelblue")
    if val_losses:
        val_steps = [steps[i] for i in range(0, len(steps), len(steps)//len(val_losses))][:len(val_losses)]
        plt.plot(val_steps, val_losses, label="Val Loss", color="coral", linestyle="--")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Survival Assistant — LoRA Training Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_loss.png", dpi=150)
    plt.show()
    print("✅ Loss plot saved to training_loss.png")

except FileNotFoundError:
    print("ℹ️  No log file found. Re-run Cell 6 with:")
    print("   !{cmd} | tee training.log")
    print("Then re-run this cell.")

In [ ]:
#%pip install --upgrade mlx-lm
#%pip install jupyterlab_wakatime

In [ ]:
import sys
import mlx_lm
from importlib.metadata import version, PackageNotFoundError

print(f"Python:  {sys.version}")
print(f"mlx-lm:  {mlx_lm.__version__}")

try:
    print(f"mlx:     {version('mlx')}")
except PackageNotFoundError:
    print("mlx:     installed, version metadata not found")

try:
    from mlx_lm.models import gemma4
    print("✅ gemma4 architecture found in mlx-lm")
except Exception as e:
    print(f"⚠️ gemma4 architecture check failed: {e}")

In [ ]:
# Quick test after training
import shlex
import sys

PROMPT = "I am lost in a cold forest with limited gear. What should I do in the first 30 minutes?"

gen_cmd = (
    f"{shlex.quote(sys.executable)} -m mlx_lm generate "
    f"--model {shlex.quote(MODEL_ID)} "
    f"--adapter-path {shlex.quote(str(ADAPTER_DIR))} "
    f"--prompt {shlex.quote(PROMPT)} "
    f"--max-tokens 220"
)

print(gen_cmd)
!{gen_cmd}